In [1]:
import pandas as pd
import spacy
import os
import numpy as np
import random
import torch
import re

In [ ]:
# Set random states.
def set_random_states(random_state):
    # Set various random seeds.
    np.random.seed(random_state)
    random.seed(random_state)
    torch.manual_seed(random_state)
    torch.cuda.manual_seed_all(random_state)
    os.environ["PYTHONHASHSEED"] = str(random_state)
    os.environ["TOKENIZERS_PARALLELISM"] = "false"
    try:
        torch.use_deterministic_algorithms(True)
    except Exception:
        pass
    return random_state

RANDOM_STATE = set_random_states(1618)

In [3]:
# Make NLP object.
nlp = spacy.load("en_core_web_sm")

In [4]:
output_dir = 'forHumanTest/Labels'

# get all real notes
all_real_notes = ((pd.read_csv('./combinedRealNotes/makeOneBigFile/dataRealAll.csv')['Note']).sample(frac=1, random_state = RANDOM_STATE)).tolist()

In [5]:
def make_analysis_files(all_fake_notes, all_real_notes, output_dir, output_file):
    names = []
    for report in all_fake_notes:
        doc = nlp(report)
        for ent in doc.ents:
            if ent.label_ == 'PERSON':
                names.append(ent.text)

    # get names for replacement in actual notes
    excluded = {'resident', 'nurse', 'therapist', 'dr', 'oedema'}
    cleaned_names = []
    for name in set(names):
        clean = name.lower()
        clean = clean.replace("'s", '').replace('mrs', '').replace('ms', '').replace('mr', '')
        
        if not any(word in clean for word in excluded):
            cleaned_names.append(clean)

    def preprocessing(text):
        text = str(text).strip()
        text = re.sub(r"\s+", " ", text)
        text = re.sub(r"^\s*-\s*", "", text) # Remove dashes at the beginning of texts.
        text = re.sub(r"^\s*\d+\.\s*", "", text) # Remove numbers in 1., 2., 3. format at the beginning of the text. 
        text = re.sub(r'\b\d+(?:\.\:|:)\s*', '', text)
        return text

    real_notes_for_test = [
        processed[0].upper() + processed[1:]
        for note in all_real_notes[:50]
        for processed in [
            preprocessing(
                note.replace('[resident]', random.choice(cleaned_names))
                    .replace('[', '')
                    .replace(']', '')
            ).strip()
        ]
    ]

    fake_notes_for_test = [
        processed[0].upper() + processed[1:]
        for note in all_fake_notes[:50]
        for processed in [
            preprocessing(
                note.replace('[resident]', random.choice(cleaned_names))
                    .replace('[', '')
                    .replace(']', '')
            ).strip()
        ]
    ]
    os.makedirs(output_dir, exist_ok=True)
    ((pd.DataFrame({'note': real_notes_for_test + fake_notes_for_test, 'label': (['real'] * len(real_notes_for_test)) + (['fake'] * len(fake_notes_for_test))}).sample(frac=1, random_state=RANDOM_STATE))).to_excel(f'./{output_dir}/{output_file}.xlsx')

In [6]:
# Online fake synthetic notes.
all_fake_notes = ((pd.read_csv('../palliativeCareNotesRealVsFake_syntheticNotesLocal/syntheticNotesOnline/allOnlineSyntheticNotes.csv')['report']).sample(frac=1, random_state = RANDOM_STATE)).tolist()

make_analysis_files(all_fake_notes, all_real_notes, output_dir, 'onlineFakeSynth')

In [7]:
# Local fake synthetic notes.
local_fake_dfs = []
for file in os.listdir('../palliativeCareNotesRealVsFake_syntheticNotesLocal/syntheticNotesLocal/fakeNoteGeneration/processedFakeNoteSyntheticNotes/'):
    df = pd.read_csv(f'../palliativeCareNotesRealVsFake_syntheticNotesLocal/syntheticNotesLocal/fakeNoteGeneration/processedFakeNoteSyntheticNotes/{file}')
    local_fake_dfs.append(df)

# get all fake notes
all_fake_notes = ((pd.concat(local_fake_dfs)['report']).sample(frac=1, random_state = RANDOM_STATE)).tolist()

make_analysis_files(all_fake_notes, all_real_notes, output_dir, 'localFakeSynth')

In [8]:
# Local real synthetic notes.
local_fake_dfs = []
for file in os.listdir('../palliativeCareNotesRealVsFake_syntheticNotesLocal/syntheticNotesLocal/realNoteGeneration/processedRealNoteSyntheticNotes/'):
    df = pd.read_csv(f'../palliativeCareNotesRealVsFake_syntheticNotesLocal/syntheticNotesLocal/realNoteGeneration/processedRealNoteSyntheticNotes/{file}')
    local_fake_dfs.append(df)

# get all fake notes
all_fake_notes = ((pd.concat(local_fake_dfs)['report']).sample(frac=1, random_state = RANDOM_STATE)).tolist()

make_analysis_files(all_fake_notes, all_real_notes, output_dir, 'localRealSynth')